In [17]:
import pandas as pd
import numpy as np

print("Environment ready")

Environment ready


In [19]:
!pip install requests

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached certifi-2026.7.22-py3-none-any.whl (136 kB)

   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   -------- ------------------------------- 1/5 [idna]
   -------- ------------------------------- 1/5 [idna]
   -------- ------------------------------- 1/5 [idna]
   -------- ------------------------------- 1/5 [idna]
   -------- ------------------------------- 1/5 [idna]
   -------- ---

In [20]:
import sys
import pandas as pd
import numpy as np
import requests

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Requests:", requests.__version__)

Python: 3.12.2 (tags/v3.12.2:6abddd9, Feb  6 2024, 21:26:36) [MSC v.1937 64 bit (AMD64)]
Pandas: 3.0.6
NumPy: 2.5.3
Requests: 2.34.2


In [ ]:
import os
import json
import requests
import pandas as pd
import world_bank_data as wb


# ============================================================
# CONFIGURACIÓN
# ============================================================

RAW_DIR = "../data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

ISO3_COUNTRIES = ["MEX", "GBR", "EST", "CHL", "COL"]
ISO2_COUNTRIES = ["MX", "GB", "EE", "CL", "CO"]

# Nombre (como lo devuelve el World Bank) -> ISO3
ISO_MAP = {
    "Chile": "CHL",
    "Colombia": "COL",
    "Estonia": "EST",
    "United Kingdom": "GBR",
    "Mexico": "MEX",
}

# Rango de años en el que se busca el último dato disponible
START_YEAR = 2020
END_YEAR = 2025

TIMEOUT = 30
HEADERS = {"User-Agent": "Digitalization-Index-Data-Project/1.0"}

print("=" * 60)
print("INICIANDO EXTRACCIÓN PROGRAMÁTICA")
print("=" * 60)


# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def save_json(data, filepath):
    """Guarda un objeto Python como JSON."""
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def get_json(url, timeout=TIMEOUT):
    """Realiza GET y valida que la respuesta sea exitosa y JSON."""
    response = requests.get(url, timeout=timeout, headers=HEADERS)
    response.raise_for_status()
    return response.json()


def get_wb_indicator(code, value_name,
                     countries=ISO3_COUNTRIES,
                     start=START_YEAR, end=END_YEAR):
    """
    Descarga un indicador del World Bank y devuelve
    todos los datos NO nulos dentro del rango de años indicado.

    Columnas de salida: country, iso3, year, <value_name>
    """

    s = wb.get_series(
        code,
        country=countries,
        date=f"{start}:{end}",
        simplify_index=True
    )

    # Nombrar la serie y convertir el índice en columnas
    df = s.rename(value_name).reset_index()
    df.columns = ["country", "year", value_name]

    # Quitar años sin dato
    df = df.dropna(subset=[value_name])

    if df.empty:
        raise ValueError(
            f"Sin datos para {code} entre {start} y {end}"
        )

    df["year"] = df["year"].astype(int)

    # Asignar ISO3
    df["iso3"] = df["country"].map(ISO_MAP)

    # Ordenar todos los años
    df = df.sort_values(
        ["country", "year"]
    ).reset_index(drop=True)

    return df[["country", "iso3", "year", value_name]]

# ============================================================
# FUENTE 1
# WORLD BANK
# (incluye TX.VAL.ICTG.ZS.UN, cuya fuente original es UNCTADstat,
#  y patentes de residentes, cuya fuente original es WIPO)
# ============================================================

print("\n[1/2] CONSULTANDO WORLD BANK API...")

wb_indicators = {
    "IT.NET.BBND.P2": "fixed_broadband_per100",
    "IT.NET.USER.ZS": "internet_users_pct",
    "TX.VAL.ICTG.ZS.UN": "ict_goods_exports_pct",     # fuente: UNCTADstat
    "GB.XPD.RSDV.GD.ZS": "rd_expenditure_gdp_pct",
    "IP.PAT.RESD": "patent_applications_resident",    # fuente: WIPO
}

wb_frames = {}

for code, name in wb_indicators.items():

    try:
        print(f"  -> Descargando {code}...")

        df = get_wb_indicator(code, name)

        output = os.path.join(RAW_DIR, f"wb_{name}.csv")
        df.to_csv(output, index=False, encoding="utf-8")

        wb_frames[name] = df

        print(f"  ✓ {name} descargado ({len(df)} registros)")
        print(df.to_string(index=False))

    except Exception as e:
        print(f"  ✗ ERROR en {code}: {e}")

# ------------------------------------------------------------
# Tabla consolidada (una fila por país, con el año de cada dato)
# ------------------------------------------------------------

if wb_frames:

    merged = pd.DataFrame({"iso3": ISO3_COUNTRIES})

    for name, df in wb_frames.items():
        part = df[["iso3", "year", name]].rename(
            columns={"year": f"{name}_year"}
        )
        merged = merged.merge(part, on="iso3", how="left")

    merged_path = os.path.join(RAW_DIR, "wb_all_indicators_latest.csv")
    merged.to_csv(merged_path, index=False, encoding="utf-8")

    print(f"\n  ✓ Tabla consolidada guardada: {merged_path}")


# ============================================================
# FUENTE 2
# ITU DATAHUB
# ============================================================

print("\n[2/2] CONSULTANDO ITU DATAHUB API...")

ITU_BASE_URL = "https://api.datahub.itu.int/v2"

try:

    # --------------------------------------------------------
    # Obtener diccionario de indicadores
    # --------------------------------------------------------

    print("  -> Descargando catálogo de indicadores ITU...")

    itu_dictionary = get_json(f"{ITU_BASE_URL}/dictionaries/getcategories")

    save_json(
        itu_dictionary,
        os.path.join(RAW_DIR, "itu_indicator_dictionary.json")
    )

    print("  ✓ Catálogo ITU guardado.")

    # --------------------------------------------------------
    # Buscar indicadores relevantes
    # --------------------------------------------------------

    def search_itu_indicators(obj, results=None):

        if results is None:
            results = []

        if isinstance(obj, dict):

            text = " ".join(
                str(obj.get(k, ""))
                for k in [
                    "name",
                    "label",
                    "indicator",
                    "indicatorName",
                    "description",
                    "Question",
                ]
            ).lower()

            if (
                "4g" in text
                or "fourth generation" in text
                or "mobile broadband" in text
            ):
                if "codeID" in obj:
                    results.append(obj)

            for value in obj.values():
                search_itu_indicators(value, results)

        elif isinstance(obj, list):

            for item in obj:
                search_itu_indicators(item, results)

        return results

    itu_candidates = search_itu_indicators(itu_dictionary)

    # Eliminar duplicados
    unique_candidates = {}

    for item in itu_candidates:
        code_id = item.get("codeID")
        if code_id is not None:
            unique_candidates[code_id] = item

    itu_candidates = list(unique_candidates.values())

    print(f"  ✓ Indicadores ITU candidatos encontrados: {len(itu_candidates)}")

    # Guardar candidatos para inspección manual
    save_json(
        itu_candidates,
        os.path.join(RAW_DIR, "itu_relevant_indicators.json")
    )

    # --------------------------------------------------------
    # Descargar datos de los indicadores encontrados
    # --------------------------------------------------------

    for indicator in itu_candidates:

        code_id = indicator.get("codeID")

        if code_id is None:
            continue

        indicator_name = (
            indicator.get("indicatorName")
            or indicator.get("label")
            or indicator.get("name")
            or f"indicator_{code_id}"
        )

        safe_name = (
            str(indicator_name)
            .lower()
            .replace(" ", "_")
            .replace("/", "_")
            .replace("%", "pct")
            .replace("(", "")
            .replace(")", "")
            .replace(",", "")
        )[:80]

        print(f"  -> ITU {code_id}: {indicator_name}")

        for iso in ISO2_COUNTRIES:

            try:
                url = f"{ITU_BASE_URL}/data/bycode/{code_id}/byiso/{iso}"

                data = get_json(url)

                output = os.path.join(
                    RAW_DIR,
                    f"itu_{code_id}_{iso}_{safe_name}.json"
                )

                save_json(data, output)

                print(f"     ✓ {iso}")

            except Exception as e:
                print(f"     ! {iso}: {e}")

except Exception as e:
    print(f"  ✗ ERROR general de ITU: {e}")


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 60)
print("EXTRACCIÓN FINALIZADA")
print("=" * 60)

print(f"\nArchivos guardados en: {os.path.abspath(RAW_DIR)}")

ModuleNotFoundError: No module named 'world_bank_data'

In [ ]:
import os
import pandas as pd


# ============================================================
# SOURCE LOG
# ============================================================

source_log_data = [

    {
        "indicador_id": "IND_01",
        "nombre_oficial": "Fixed broadband subscriptions (per 100 people)",
        "categoria": "Infraestructura",
        "fuente": "World Bank",
        "url_origen": "https://data.worldbank.org/indicator/IT.NET.BBND.P2",
        "unidad": "Por cada 100 hab",
        "metodo_obtencion": "API Python (world_bank_data)",
        "archivo_raw": "wb_fixed_broadband_per100.csv"
    },

    {
        "indicador_id": "IND_02",
        "nombre_oficial": "4G mobile network coverage (% of population)",
        "categoria": "Infraestructura",
        "fuente": "ITU DataHub",
        "url_origen": "https://datahub.itu.int/",
        "unidad": "% población",
        "metodo_obtencion": "API REST JSON (requests)",
        "archivo_raw": "itu_*_*.json"
    },

    {
        "indicador_id": "IND_03",
        "nombre_oficial": "Individuals using the Internet (% of population)",
        "categoria": "Acceso / Uso",
        "fuente": "World Bank",
        "url_origen": "https://data.worldbank.org/indicator/IT.NET.USER.ZS",
        "unidad": "% población",
        "metodo_obtencion": "API Python (world_bank_data)",
        "archivo_raw": "wb_internet_users_pct.csv"
    },

    {
        "indicador_id": "IND_04",
        "nombre_oficial": "Mobile broadband basket (% of GNI per capita)",
        "categoria": "Asequibilidad",
        "fuente": "ITU DataHub",
        "url_origen": "https://datahub.itu.int/",
        "unidad": "% GNI per capita",
        "metodo_obtencion": "API REST JSON (requests)",
        "archivo_raw": "itu_*_*.json"
    },

    {
        "indicador_id": "IND_05",
        "nombre_oficial": "ICT goods exports (% of total goods exports)",
        "categoria": "Económica Digital",
        "fuente": "World Bank / UNCTADstat",
        "url_origen": "https://data.worldbank.org/indicator/TX.VAL.ICTG.ZS.UN",
        "unidad": "% exportaciones",
        "metodo_obtencion": "API Python (world_bank_data)",
        "archivo_raw": "wb_ict_goods_exports_pct.csv"
    },

    {
        "indicador_id": "IND_06",
        "nombre_oficial": "Digitally deliverable services exports",
        "categoria": "Económica Digital",
        "fuente": "UNCTADstat",
        "url_origen": "https://unctadstat.unctad.org/",
        "unidad": "Según unidad publicada por UNCTADstat",
        "metodo_obtencion": "Dataset oficial UNCTADstat",
        "archivo_raw": "Pendiente de extracción directa"
    },

    {
        "indicador_id": "IND_07",
        "nombre_oficial": "Patent applications, residents",
        "categoria": "Innovación",
        "fuente": "World Bank / WIPO",
        "url_origen": "https://data.worldbank.org/indicator/IP.PAT.REGS",
        "unidad": "Número de solicitudes",
        "metodo_obtencion": "API Python (world_bank_data)",
        "archivo_raw": "wipo_patents.csv"
    },

    {
        "indicador_id": "IND_08",
        "nombre_oficial": "Research and development expenditure (% of GDP)",
        "categoria": "Innovación",
        "fuente": "World Bank",
        "url_origen": "https://data.worldbank.org/indicator/GB.XPD.RSDV.GD.ZS",
        "unidad": "% del PIB",
        "metodo_obtencion": "API Python (world_bank_data)",
        "archivo_raw": "wb_rd_expenditure_gdp_pct.csv"
    }
]


# ============================================================
# CREAR DATAFRAME
# ============================================================

df_log = pd.DataFrame(source_log_data)


# ============================================================
# AGREGAR AÑO REAL DE OBSERVACIÓN
# ============================================================

df_log["ano_observacion"] = None


def get_latest_year(filepath):
    """
    Obtiene el año más reciente disponible de un CSV
    descargado desde World Bank.
    """

    if not os.path.exists(filepath):
        return None

    try:
        df = pd.read_csv(filepath)

        if "year" not in df.columns:
            return None

        years = pd.to_numeric(
            df["year"],
            errors="coerce"
        ).dropna()

        if years.empty:
            return None

        return int(years.max())

    except Exception:
        return None


# World Bank
df_log.loc[
    df_log["indicador_id"] == "IND_01",
    "ano_observacion"
] = get_latest_year(
    "../data/raw/wb_fixed_broadband_per100.csv"
)

df_log.loc[
    df_log["indicador_id"] == "IND_03",
    "ano_observacion"
] = get_latest_year(
    "../data/raw/wb_internet_users_pct.csv"
)

df_log.loc[
    df_log["indicador_id"] == "IND_05",
    "ano_observacion"
] = get_latest_year(
    "../data/raw/wb_ict_goods_exports_pct.csv"
)

df_log.loc[
    df_log["indicador_id"] == "IND_07",
    "ano_observacion"
] = get_latest_year(
    "../data/raw/wipo_patents.csv"
)

df_log.loc[
    df_log["indicador_id"] == "IND_08",
    "ano_observacion"
] = get_latest_year(
    "../data/raw/wb_rd_expenditure_gdp_pct.csv"
)


# ============================================================
# GUARDAR LOG
# ============================================================

output_path = "../source_log.csv"

df_log.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"✓ Archivo source_log.csv generado: "
    f"{os.path.abspath(output_path)}"
)

print("\nResumen del source log:")
print(
    df_log[
        [
            "indicador_id",
            "nombre_oficial",
            "fuente",
            "ano_observacion"
        ]
    ].to_string(index=False)
)

In [ ]:
# %% [markdown]
# # Digital Economy Intelligence · Preparación de datos
#
# **Equipo 2:** México, Reino Unido, Estonia, Chile y Colombia · **Sector:** Salud
#
# Este bloque cubre: lectura de los archivos crudos (`data/raw/`, sin modificarlos), armonización de tres estructuras distintas
# (World Bank, ITU, UNCTADstat), selección documentada del año de observación, exploración de faltantes y generación de
# `digital_economy_clean.csv`, `source_log.csv` y `data_dictionary.csv`.
#
# El notebook localiza solo la raíz del proyecto (la carpeta que contiene `data/raw/`), esté en la raíz o en `notebooks/`.
# Guarda en `data/processed/` y deja `source_log.csv` y `data_dictionary.csv` en la raíz.

# %% [markdown]
# ## 1. Configuración

# %%
import os
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:          # al ejecutar como script simple
    display = print

def find_project_root():
    """Sube desde la carpeta actual hasta encontrar la que contiene data/raw.
    Así funciona igual si el notebook está en la raíz o dentro de notebooks/."""
    here = os.path.abspath(os.getcwd())
    p = here
    while True:
        if os.path.isdir(os.path.join(p, "data", "raw")):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            raise FileNotFoundError(
                f"No se encontró data/raw/ desde {here} ni en sus carpetas superiores")
        p = parent


ROOT = find_project_root()
RAW_DIR = os.path.join(ROOT, "data", "raw")
PROC_DIR = os.path.join(ROOT, "data", "processed")
os.makedirs(PROC_DIR, exist_ok=True)

print("Carpeta de trabajo :", os.getcwd())
print("Raíz del proyecto  :", ROOT)
print("Datos crudos en    :", RAW_DIR)

COUNTRIES = ["MEX", "GBR", "EST", "CHL", "COL"]

# Unifica los nombres de país que usa cada fuente -> ISO3
ISO_MAP = {
    "Mexico": "MEX", "México": "MEX",
    "United Kingdom": "GBR", "UK": "GBR",
    "Estonia": "EST",
    "Chile": "CHL",
    "Colombia": "COL",
}
COUNTRY_NAMES = {"MEX": "México", "GBR": "Reino Unido", "EST": "Estonia",
                 "CHL": "Chile", "COL": "Colombia"}

# Año objetivo de comparación (se usa si existe para los 5 países)
TARGET_YEAR = 2024

# Máximo de años de rezago aceptado respecto al año objetivo.
# Un dato anterior a (TARGET_YEAR - MAX_LAG) se considera NO comparable y se deja como no disponible.
MAX_LAG = 4

# %% [markdown]
# ## 2. Funciones de lectura (una por estructura)
#
# | Fuente | Estructura del archivo | Particularidades |
# |---|---|---|
# | World Bank | `country, iso3, year, <valor>` | ya viene limpio (generado por `01_extraccion_wb.py`) |
# | ITU DataHub | `Economy, 2023, 2024, ...` (ancho) | valores como texto con `%`; `-` significa **dato no disponible** |
# | UNCTADstat | `Economy_Label, Year, <valor>` (largo) | nombre de país sin ISO3 |
#
# Todas devuelven el mismo formato largo: `iso3, year, value`. Un `-` se convierte en `NaN`, **nunca en 0**.

# %%
def resolve(ind_id, default):
    """Busca en RAW_DIR un CSV que empiece con el ID (ej. IND_01*.csv, sin importar
    mayúsculas). Si no existe, usa el nombre esperado por defecto."""
    if os.path.isdir(RAW_DIR):
        for f in sorted(os.listdir(RAW_DIR)):
            if f.lower().startswith(ind_id.lower()) and f.lower().endswith(".csv"):
                return f
    return default


def _ensure_iso3(df):
    if "iso3" not in df.columns and "country" in df.columns:
        df["iso3"] = df["country"].astype(str).str.strip().map(ISO_MAP)
    return df


def _finalize(df):
    """Filtra a los 5 países y avisa si alguna fila no se pudo mapear a ISO3."""
    unmapped = df[df["iso3"].isna()]
    if len(unmapped):
        print("  ! Filas sin ISO3 (revisar ISO_MAP):", len(unmapped))
    df = df[df["iso3"].isin(COUNTRIES)].copy()
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    return df.dropna(subset=["year"])[["iso3", "year", "value"]]


def load_wb(fname, value_col):
    df = _ensure_iso3(pd.read_csv(os.path.join(RAW_DIR, fname)))
    if value_col not in df.columns:      # p. ej. si renombraste la columna
        value_col = df.columns[3]
    df = df.rename(columns={value_col: "value"})
    if df.groupby("iso3").size().max() == 1:
        print(f"  ! {fname} solo trae UN año por país (el último disponible). "
              "Vuelve a correr 01_extraccion_wb.py para bajar la serie completa "
              "y poder elegir un año común.")
    return _finalize(df[["iso3", "year", "value"]])


def load_itu(fname):
    """Formato ancho con años como columnas, '%' como texto y '-' como faltante."""
    df = pd.read_csv(os.path.join(RAW_DIR, fname), dtype=str)
    econ = df.columns[0]
    long = df.melt(id_vars=econ, var_name="year", value_name="value")
    long["iso3"] = long[econ].str.strip().map(ISO_MAP)
    v = long["value"].str.strip()
    v = v.replace({"-": np.nan, "": np.nan, "..": np.nan})
    v = v.str.replace("%", "", regex=False).str.replace(",", "", regex=False)
    long["value"] = pd.to_numeric(v, errors="coerce")
    return _finalize(long)


def load_unctad(fname):
    """Formato largo: país, año, valor (las tres primeras columnas)."""
    df = pd.read_csv(os.path.join(RAW_DIR, fname))
    econ, year, val = df.columns[:3]
    out = pd.DataFrame({
        "iso3": df[econ].astype(str).str.strip().map(ISO_MAP),
        "year": df[year],
        "value": pd.to_numeric(df[val], errors="coerce"),
    })
    return _finalize(out)


def ensure_population(population_file):
    """Devuelve la ruta de la población total. Si no está en data/raw/, la descarga
    del World Bank (SP.POP.TOTL) y la guarda ahí. Solo se usa como denominador."""
    path = os.path.join(RAW_DIR, population_file)
    if os.path.exists(path):
        return path
    print("  ! No está", population_file, "-> descargando SP.POP.TOTL del World Bank...")
    try:
        import world_bank_data as wb
        s = wb.get_series("SP.POP.TOTL", country=COUNTRIES,
                          date="2015:2025", simplify_index=True)
        df = s.rename("population_total").reset_index()
        df.columns = ["country", "year", "population_total"]
        df = df.dropna(subset=["population_total"])
        df["year"] = df["year"].astype(int)
        df["iso3"] = df["country"].map(ISO_MAP)
        df[["country", "iso3", "year", "population_total"]].to_csv(
            path, index=False, encoding="utf-8")
        print("  ✓ población guardada en", path)
    except Exception as e:
        raise FileNotFoundError(
            2, f"No se pudo obtener la población (SP.POP.TOTL): {e}", path)
    return path


def load_patents_per_million(patents_file, population_file):
    """Patentes de residentes / población del MISMO año x 1,000,000."""
    pat = _ensure_iso3(pd.read_csv(os.path.join(RAW_DIR, patents_file)))
    pcol = ("patent_applications_resident"
            if "patent_applications_resident" in pat.columns else pat.columns[3])
    pat = pat.rename(columns={pcol: "patents"})
    if pat.groupby("iso3").size().max() == 1:
        print(f"  ! {patents_file} solo trae UN año por país. "
              "Vuelve a correr 01_extraccion_wb.py para bajar la serie completa.")
    pop = pd.read_csv(ensure_population(population_file))
    m = pat.merge(pop[["iso3", "year", "population_total"]],
                  on=["iso3", "year"], how="left")
    sin_pob = m[m["population_total"].isna()]
    if len(sin_pob):
        print("  ! Patentes sin población del mismo año:",
              sin_pob[["iso3", "year"]].values.tolist())
    m["value"] = m["patents"] / m["population_total"] * 1_000_000
    return _finalize(m[["iso3", "year", "value"]])

# %% [markdown]
# ## 3. Registro de los 8 indicadores
#
# Cada indicador tiene su categoría (según la distribución exigida en la actividad), unidad, sentido y transformación.
# Las definiciones son breves: **verifícalas contra los metadatos de la fuente** antes de entregar.

# %%
INDICATORS = [
    dict(id="IND_01", col="fixed_broadband_per100", category="Infraestructura",
         name="Fixed broadband subscriptions (per 100 people)",
         definition="Suscripciones a banda ancha fija por cada 100 habitantes.",
         source="World Bank (WDI)", url="https://data.worldbank.org/indicator/IT.NET.BBND.P2",
         unit="Por cada 100 hab", direction="mayor es mejor", method="API Python (world_bank_data)",
         raw="wb_fixed_broadband_per100.csv", transformation="Ninguna",
         loader=lambda: load_wb(resolve("IND_01", "wb_fixed_broadband_per100.csv"), "fixed_broadband_per100")),

    dict(id="IND_02", col="coverage_4g_pct", category="Infraestructura",
         name="4G mobile network coverage (% of population)",
         definition="Porcentaje de la población cubierta por red móvil 4G.",
         source="ITU DataHub", url="https://datahub.itu.int/data/",
         unit="% población", direction="mayor es mejor", method="Descarga manual CSV",
         raw="itu_4g_coverage.csv", transformation="Quitar '%' y convertir '-' a NaN",
         loader=lambda: load_itu(resolve("IND_02", "itu_4g_coverage.csv"))),

    dict(id="IND_03", col="internet_users_pct", category="Acceso / Uso",
         name="Individuals using the Internet (% of population)",
         definition="Porcentaje de la población que usó internet en los últimos 3 meses.",
         source="World Bank (WDI)", url="https://data.worldbank.org/indicator/IT.NET.USER.ZS",
         unit="% población", direction="mayor es mejor", method="API Python (world_bank_data)",
         raw="wb_internet_users_pct.csv", transformation="Ninguna",
         loader=lambda: load_wb(resolve("IND_03", "wb_internet_users_pct.csv"), "internet_users_pct")),

    dict(id="IND_04", col="mobile_bb_basket_pct_gni", category="Calidad / Asequibilidad",
         name="Mobile broadband basket (% of GNI per capita)",
         definition="Costo de la canasta de banda ancha móvil como % del INB per cápita.",
         source="ITU DataHub", url="https://datahub.itu.int/data/",
         unit="% GNI per capita", direction="menor es mejor (inverso)", method="Descarga manual CSV",
         raw="itu_mobile_broadband_basket.csv", transformation="Quitar '%' y convertir '-' a NaN",
         loader=lambda: load_itu(resolve("IND_04", "itu_mobile_broadband_basket.csv"))),

    dict(id="IND_05", col="ict_goods_exports_pct", category="Actividad económica digital",
         name="ICT goods exports (% of total goods exports)",
         definition="Exportaciones de bienes TIC como % del total de exportaciones de bienes.",
         source="World Bank (WDI) / UNCTADstat", url="https://data.worldbank.org/indicator/TX.VAL.ICTG.ZS.UN",
         unit="% exportaciones de bienes", direction="mayor es mejor", method="API Python (world_bank_data)",
         raw="wb_ict_goods_exports_pct.csv", transformation="Ninguna",
         loader=lambda: load_wb(resolve("IND_05", "wb_ict_goods_exports_pct.csv"), "ict_goods_exports_pct")),

    dict(id="IND_06", col="digital_services_pct_trade", category="Actividad económica digital",
         name="Digitally deliverable services (% of total trade in services)",
         definition="Servicios digitalmente entregables como % del comercio total de servicios "
                    "(confirmar en el archivo de UNCTADstat si es exportación, importación o total).",
         source="UNCTADstat", url="https://unctadstat.unctad.org/",
         unit="% del comercio de servicios", direction="mayor es mejor", method="Descarga manual CSV",
         raw="unctad_digitally_deliverable_services.csv", transformation="Ninguna (ya es proporción)",
         loader=lambda: load_unctad(resolve("IND_06", "unctad_digitally_deliverable_services.csv"))),

    dict(id="IND_07", col="patents_res_per_million", category="Capacidad tecnológica / Innovación",
         name="Patent applications, residents (per million inhabitants)",
         definition="Solicitudes de patente de residentes por millón de habitantes.",
         source="World Bank (WDI) / WIPO", url="https://data.worldbank.org/indicator/IP.PAT.RESD",
         unit="Solicitudes por millón de hab", direction="mayor es mejor", method="API Python (world_bank_data)",
         raw="wb_patent_applications_resident.csv + wb_population_total.csv",
         transformation="Solicitudes / población total del mismo año x 1,000,000 (población: WB SP.POP.TOTL)",
         loader=lambda: load_patents_per_million(resolve("IND_07", "wb_patent_applications_resident.csv"),
                                                 "wb_population_total.csv")),

    dict(id="IND_08", col="rd_expenditure_gdp_pct", category="Capacidad tecnológica / Innovación",
         name="Research and development expenditure (% of GDP)",
         definition="Gasto en investigación y desarrollo como % del PIB.",
         source="World Bank (WDI)", url="https://data.worldbank.org/indicator/GB.XPD.RSDV.GD.ZS",
         unit="% del PIB", direction="mayor es mejor", method="API Python (world_bank_data)",
         raw="wb_rd_expenditure_gdp_pct.csv", transformation="Ninguna",
         loader=lambda: load_wb(resolve("IND_08", "wb_rd_expenditure_gdp_pct.csv"), "rd_expenditure_gdp_pct")),
]
assert len(INDICATORS) == 8, "La actividad exige exactamente 8 indicadores"

# %% [markdown]
# ## 4. Carga de los 8 indicadores (formato largo)

# %%
frames, pendientes = [], []

for ind in INDICATORS:
    try:
        d = ind["loader"]()
        d["indicator_id"] = ind["id"]
        frames.append(d)
        print(f"✓ {ind['id']} {ind['col']:<28} filas: {len(d)}")
    except Exception as e:
        pendientes.append(ind["id"])
        detalle = getattr(e, "filename", None) or e
        print(f"✗ {ind['id']} {type(e).__name__}: {detalle}")

long = pd.concat(frames, ignore_index=True)
print("\nIndicadores pendientes:", pendientes or "ninguno")

if pendientes:
    print("\nArchivos que SÍ hay en", os.path.abspath(RAW_DIR), ":")
    print(sorted(os.listdir(RAW_DIR)) if os.path.isdir(RAW_DIR)
          else "  (la carpeta no existe: abre el notebook desde la raíz del proyecto)")
long.head(10)

# %% [markdown]
# ## 5. Disponibilidad por indicador, país y año
#
# Antes de elegir el año hay que ver **qué existe**. Un `NaN` significa dato no disponible; un `0` es un valor real.

# %%
disp = (long.dropna(subset=["value"])
            .groupby(["indicator_id", "iso3"])["year"]
            .apply(lambda s: sorted(int(y) for y in s))
            .unstack("iso3")
            .reindex(columns=COUNTRIES))
disp

# %% [markdown]
# ## 6. Selección del año de observación
#
# Objetivo: comparar los 5 países **en el mismo año** siempre que se pueda, y reportar con claridad cuando no.
# Reglas, por indicador (nunca se usan años posteriores a `TARGET_YEAR`):
# 1. Si `TARGET_YEAR` tiene dato para los 5 países, se usa ese año.
# 2. Si no, se usa el **último año común** a los 5 países, siempre que no tenga más de `MAX_LAG` años de rezago.
# 3. Si no hay año común dentro de esa tolerancia, se toma el último dato de cada país dentro de la ventana
#    `[TARGET_YEAR - MAX_LAG, TARGET_YEAR]` y el indicador queda marcado como **años mezclados**.
#    Un país sin dato dentro de la ventana queda como **no disponible (NaN)**, no se rellena con un dato viejo.

# %%
def pick_year(g, target=TARGET_YEAR, max_lag=MAX_LAG):
    ok = g.dropna(subset=["value"])
    ok = ok[ok["year"] <= target]
    sets = [set(ok.loc[ok["iso3"] == c, "year"].astype(int)) for c in COUNTRIES]
    common = set.intersection(*sets)
    if target in common:
        return target, "año objetivo común a los 5 países"
    cerca = {y for y in common if y >= target - max_lag}
    if cerca:
        return max(cerca), f"último año común (rezago ≤ {max_lag} años)"
    return None, "sin año común dentro de la tolerancia: años mezclados"


rows, sel = [], []

for ind in INDICATORS:
    if ind["id"] in pendientes:
        continue
    g = long[long["indicator_id"] == ind["id"]].dropna(subset=["value"])
    y, criterio = pick_year(g)
    usados = []
    for c in COUNTRIES:
        gc = g[(g["iso3"] == c) & (g["year"] <= TARGET_YEAR)].sort_values("year")
        r = gc[gc["year"] == y] if y is not None else gc[gc["year"] >= TARGET_YEAR - MAX_LAG].tail(1)
        val = r["value"].iloc[0] if len(r) else np.nan
        yr_ = int(r["year"].iloc[0]) if len(r) else np.nan
        usados.append(yr_)
        rows.append({"iso3": c, "col": ind["id"], "value": val, "year": yr_})

    validos = [u for u in usados if pd.notna(u)]
    sin_dato = [c for c, u in zip(COUNTRIES, usados) if pd.isna(u)]
    if validos:
        lo, hi = int(min(validos)), int(max(validos))
        y_txt = str(lo) if lo == hi else f"{lo}-{hi}"
        rezago = TARGET_YEAR - lo
    else:
        y_txt, rezago = "sin datos", np.nan

    if sin_dato:
        estado = "ALERTA: faltan países"
    elif y is None:
        estado = "AVISO: años mezclados"
    elif y < TARGET_YEAR:
        estado = f"AVISO: año común {y} (< {TARGET_YEAR})"
    else:
        estado = "OK"

    sel.append({"indicator_id": ind["id"], "nombre": ind["name"],
                "year_used": y_txt, "criterio": criterio,
                "paises_sin_dato": ", ".join(sin_dato) or "-",
                "rezago_max_vs_objetivo": rezago, "estado": estado})

year_selection = pd.DataFrame(sel)
year_selection[["indicator_id", "year_used", "estado", "paises_sin_dato", "criterio"]]

# %% [markdown]
# ## 7. Dataset integrado (5 economías × 8 indicadores)

# %%
sel_long = pd.DataFrame(rows)

values = sel_long.pivot(index="iso3", columns="col", values="value").reindex(COUNTRIES)
years = (sel_long.pivot(index="iso3", columns="col", values="year")
                 .reindex(COUNTRIES).add_suffix("_year"))

years = years.astype("Int64")
clean = pd.concat([values, years], axis=1)
clean.insert(0, "country", clean.index.map(COUNTRY_NAMES))
clean = clean.reset_index()

# Orden de columnas: valor e año de cada indicador, juntos
ordered = ["iso3", "country"]
for ind in INDICATORS:
    if ind["id"] in clean.columns:
        ordered += [ind["id"], ind["id"] + "_year"]
clean = clean[ordered]
clean

# %% [markdown]
# ## 8. Exploración y validaciones

# %%
value_cols = [i["id"] for i in INDICATORS if i["id"] in clean.columns]
print({i["id"]: i["name"] for i in INDICATORS}, "\n")

print("Forma del dataset:", clean.shape, "(esperado: 5 filas)")
print("Indicadores presentes:", len(value_cols), "de 8\n")

print("Datos NO disponibles (NaN) por indicador:")
print(clean[value_cols].isna().sum().to_string())

print("\nValores iguales a CERO (dato real, distinto de faltante):")
print((clean[value_cols] == 0).sum().to_string())

print("\nEstadísticos descriptivos:")
clean[value_cols].describe().T[["min", "mean", "max", "std"]].round(2)

# %%
# --- Revisión de años: ¿qué año quedó en cada celda de la tabla final? ---
year_cols = [c + "_year" for c in value_cols]
matriz_anos = clean.set_index("iso3")[year_cols].copy()
matriz_anos.columns = value_cols
print("Año de observación usado (país × indicador):")
display(matriz_anos)

print("\nIndicadores que NO están todos en el año objetivo", TARGET_YEAR, ":")
fuera = year_selection[year_selection["estado"] != "OK"]
display(fuera[["indicator_id", "year_used", "estado", "paises_sin_dato"]]
        if len(fuera) else "ninguno")

print("\nAños distintos dentro de un mismo indicador (comparación menos directa):")
mezclados = [c for c in value_cols if matriz_anos[c].nunique(dropna=True) > 1]
print(mezclados or "ninguno")

# %% [markdown]
# **Escribe aquí tus decisiones de preparación** (la actividad exige documentarlas):
#
# - Año elegido por indicador y por qué:
# - Faltantes encontrados y tratamiento:
# - Valores extremos o inconsistencias detectadas:
# - Transformaciones aplicadas (ej. patentes por millón de habitantes):
# - Cuándo comparar años distintos puede resultar engañoso:

# %% [markdown]
# ## 9. Guardado: dataset limpio, source log y data dictionary

# %%
clean.to_csv(os.path.join(PROC_DIR, "digital_economy_clean.csv"),
             index=False, encoding="utf-8-sig")
year_selection.to_csv(os.path.join(PROC_DIR, "year_selection.csv"),
                      index=False, encoding="utf-8-sig")

# ---------- Source log ----------
yr = year_selection.set_index("indicator_id")["year_used"].to_dict()
crit = year_selection.set_index("indicator_id")["criterio"].to_dict()

source_log = pd.DataFrame([{
    "indicador_id": i["id"],
    "nombre_oficial": i["name"],
    "definicion": i["definition"],
    "categoria": i["category"],
    "fuente": i["source"],
    "url_origen": i["url"],
    "unidad": i["unit"],
    "ano_observacion": yr.get(i["id"]),
    "criterio_ano": crit.get(i["id"]),
    "metodo_obtencion": i["method"],
    "archivo_raw": i["raw"],
    "transformacion": i["transformation"],
} for i in INDICATORS])
source_log.to_csv(os.path.join(ROOT, "source_log.csv"), index=False, encoding="utf-8-sig")

# ---------- Data dictionary ----------
data_dictionary = pd.DataFrame([{
    "variable": i["id"],
    "nombre_indicador": i["name"],
    "indicador_id": i["id"],
    "definicion": i["definition"],
    "unidad": i["unit"],
    "categoria": i["category"],
    "sentido": i["direction"],
    "fuente": i["source"],
} for i in INDICATORS] + [{
    "variable": f"{i['id']}_year", "nombre_indicador": i["name"], "indicador_id": i["id"],
    "definicion": f"Año de observación de {i['id']}", "unidad": "año",
    "categoria": "Metadato", "sentido": "n/a", "fuente": i["source"],
} for i in INDICATORS])
data_dictionary.to_csv(os.path.join(ROOT, "data_dictionary.csv"), index=False, encoding="utf-8-sig")

print("✓ digital_economy_clean.csv, year_selection.csv, source_log.csv y data_dictionary.csv generados")
source_log[["indicador_id", "fuente", "ano_observacion", "criterio_ano"]]

# %% [markdown]
# ## Siguiente paso
# Con `digital_economy_clean.csv` listo: (1) fundamentación conceptual, (2) cálculo del **DRS** (normalización, sentido inverso de
# `IND_04`, pesos que sumen 1 y regla para faltantes), (3) visualizaciones y (4) dashboard.

Carpeta de trabajo : c:\users\ramir\github_repos\digital-economy\notebooks\.ipynb_checkpoints
Raíz del proyecto  : c:\users\ramir\github_repos\digital-economy
Datos crudos en    : c:\users\ramir\github_repos\digital-economy\data\raw
  ! IND_01.csv solo trae UN año por país (el último disponible). Vuelve a correr 01_extraccion_wb.py para bajar la serie completa y poder elegir un año común.
✓ IND_01 fixed_broadband_per100       filas: 5
✓ IND_02 coverage_4g_pct              filas: 20
  ! IND_03.csv solo trae UN año por país (el último disponible). Vuelve a correr 01_extraccion_wb.py para bajar la serie completa y poder elegir un año común.
✓ IND_03 internet_users_pct           filas: 5
✓ IND_04 mobile_bb_basket_pct_gni     filas: 20
  ! IND_05.csv solo trae UN año por país (el último disponible). Vuelve a correr 01_extraccion_wb.py para bajar la serie completa y poder elegir un año común.
✓ IND_05 ict_goods_exports_pct        filas: 5
✓ IND_06 digital_services_pct_trade   filas: 15
  ! w

,IND_01,IND_02,IND_03,IND_04,IND_05,IND_06,IND_08
iso3,,,,,,,
MEX,2024,2024,2024,2024,2024,2024,2024
GBR,2024,2024,2024,2024,2024,2024,2023
EST,2024,2024,2024,2024,2024,2024,2023
CHL,2024,2024,2024,2024,2024,2024,2022
COL,2024,2024,2024,2024,2024,2024,2020



Indicadores que NO están todos en el año objetivo 2024 :


,indicator_id,year_used,estado,paises_sin_dato
6,IND_08,2020-2024,AVISO: años mezclados,-



Años distintos dentro de un mismo indicador (comparación menos directa):
['IND_08']
✓ digital_economy_clean.csv, year_selection.csv, source_log.csv y data_dictionary.csv generados


,indicador_id,fuente,ano_observacion,criterio_ano
0,IND_01,World Bank (WDI),2024,año objetivo común a los 5 países
1,IND_02,ITU DataHub,2024,año objetivo común a los 5 países
2,IND_03,World Bank (WDI),2024,año objetivo común a los 5 países
3,IND_04,ITU DataHub,2024,año objetivo común a los 5 países
4,IND_05,World Bank (WDI) / UNCTADstat,2024,año objetivo común a los 5 países
5,IND_06,UNCTADstat,2024,año objetivo común a los 5 países
6,IND_07,World Bank (WDI) / WIPO,NaN,NaN
7,IND_08,World Bank (WDI),2020-2024,sin año común dentro de la tolerancia: años me...
